# S6E11 GPU · Trung bình thứ hạng LightGBM và XGBoost

Thử nghiệm trên dữ liệu chính thức của cuộc thi. Cách chia 5 fold và seed 42 giống đợt trước để so sánh trực tiếp.

## 1. Dữ liệu và môi trường GPU

Nguồn nhãn là `train.csv` chính thức của S6E11. Mã kiểm tra SHA-256 phải khớp
bản dữ liệu của repo. Tập kiểm tra của Kaggle không có nhãn và ROC AUC trong
notebook được đo trên các giao dịch giữ lại từ tập huấn luyện có nhãn.
`id` chỉ dùng để truy vết. Mỗi mô hình xác nhận thiết bị GPU trước khi fit.

In [1]:
import hashlib
import json
import os
from pathlib import Path
from subprocess import check_output
import sys
from time import perf_counter
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
SMOKE = os.environ.get('S6E11_SMOKE') == '1'
ROOT = Path(os.environ.get(
    'S6E11_PROJECT', r'D:\UET\ML\Predicting Credit Card Fraud'
)).resolve(strict=True)
REPO = ROOT / 'INT3405E_FD'
CONFIG = json.loads((REPO / 'configs' / 'default.json').read_text(encoding='utf-8-sig'))
MANIFEST = json.loads((REPO / 'data' / 'manifest.json').read_text(encoding='utf-8-sig'))
TARGET = CONFIG['target']
NUMERIC = CONFIG['numeric_columns']
FLAGS = CONFIG['flag_columns']
CATEGORICAL = CONFIG['categorical_columns']

GPU_NAME = check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        text=True, timeout=15).splitlines()[0].strip()
if 'RTX 4060' not in GPU_NAME:
    raise RuntimeError(f'GPU hiện tại khác RTX 4060 đã kiểm tra: {GPU_NAME}')
print(f'GPU phát hiện: {GPU_NAME}')

def load_competition_train():
    expected_hash = MANIFEST['input_files']['train.csv']['sha256']
    csv_path = ROOT / 'train.csv'
    if csv_path.is_file():
        with csv_path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv không khớp manifest')
        return pd.read_csv(csv_path), digest

    with ZipFile(ROOT / 'playground-series-s6e11.zip') as archive:
        with archive.open('train.csv') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv trong ZIP không khớp manifest')
        with archive.open('train.csv') as stream:
            return pd.read_csv(stream), digest

frame, DATA_HASH = load_competition_train()
feature_names = NUMERIC + FLAGS + CATEGORICAL
required = ['id', TARGET, *feature_names]
if (set(frame.columns) != set(required) or len(frame) != MANIFEST['train_rows']
        or frame.id.duplicated().any() or frame[required].isna().any().any()
        or set(frame[TARGET].unique()) != {0, 1}):
    raise ValueError('Dữ liệu huấn luyện không khớp schema đã kiểm tra của repo')
if SMOKE:
    _, frame = train_test_split(
        frame, test_size=20_000, stratify=frame[TARGET], random_state=SEED,
    )
    frame = frame.reset_index(drop=True)
y = frame[TARGET].to_numpy(dtype=np.uint8)
ids = frame['id'].to_numpy(dtype=np.int64)
X = frame[feature_names]
raw_numeric = X[NUMERIC + FLAGS].to_numpy(dtype=np.float32)
if not np.isfinite(raw_numeric).all():
    raise ValueError('Đặc trưng số có giá trị không hữu hạn')
print(f'Số dòng: {len(y):,} | nhãn gian lận: {y.sum():,} ({y.mean():.2%})')

ROUND2_DIR = ROOT / 's6e11_gpu' / 'round2'

GPU phát hiện: NVIDIA GeForce RTX 4060 Laptop GPU
Số dòng: 699,960 | nhãn gian lận: 11,689 (1.67%)


## 2. Năm phần chia phân tầng

Dùng `StratifiedKFold` với seed 42 như repo để giữ tỷ lệ gian lận giữa các phần.
Mỗi giao dịch được chấm một lần bởi mô hình học từ bốn phần còn lại.
Số vòng hoặc số epoch được cố định trước khi đánh giá phần giữ lại.

In [2]:
n_folds = 2 if SMOKE else 5
splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
splits = list(splitter.split(X, y))
assignment = np.full(len(y), -1, dtype=np.int8)
for fold, (_, val_idx) in enumerate(splits):
    assignment[val_idx] = fold
if (assignment < 0).any():
    raise RuntimeError('Có giao dịch chưa được gán fold')
SPLIT_HASH = hashlib.sha256(ids.astype('<i8').tobytes() + assignment.tobytes()).hexdigest()
print(f'{n_folds} fold | mã kiểm tra cách chia: {SPLIT_HASH[:16]}')

5 fold | mã kiểm tra cách chia: e3629fd42119a82d


## 3. Đọc dự đoán ngoài phần huấn luyện của hai mô hình cây

Mỗi điểm lấy từ mô hình GPU chưa học giao dịch tương ứng. Hai notebook đối
chứng trong `.build/reference_notebooks` tái chạy đúng mã mô hình đợt trước
để lưu dự đoán. Cell này kiểm tra mã dữ liệu, cách chia và ID từng giao dịch
trước khi ghép kết quả.

In [3]:
import torch

if SMOKE:
    raise RuntimeError('Notebook kết hợp dùng các tệp OOF đủ 5 fold')
if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != GPU_NAME:
    raise RuntimeError('CUDA trên RTX 4060 chưa sẵn sàng')

reference_dir = ROUND2_DIR / 'results' / 'references'

def load_reference(slug):
    path = reference_dir / f'{slug}-oof.npz'
    with np.load(path, allow_pickle=False) as saved:
        if (saved['train_sha256'].item() != DATA_HASH
                or saved['split_sha256'].item() != SPLIT_HASH
                or not np.array_equal(saved['ids'], ids)
                or not np.array_equal(saved['fold'], assignment)):
            raise ValueError(f'Dữ liệu hoặc thứ tự giao dịch sai: {path.name}')
        scores = saved['prediction']
    if (scores.shape != ids.shape or not np.isfinite(scores).all()
            or (scores < 0).any() or (scores > 1).any()):
        raise ValueError(f'Dự đoán không hợp lệ: {path.name}')
    info = json.loads((reference_dir / f'{slug}.json').read_text(encoding='utf-8'))
    print(f"{slug}: {len(scores):,} dự đoán OOF")
    return scores, info

lightgbm_oof, lightgbm_metrics = load_reference('lightgbm-te')
xgboost_oof, xgboost_metrics = load_reference('xgboost-te')

lightgbm-te: 699,960 dự đoán OOF
xgboost-te: 699,960 dự đoán OOF


## 4. Lấy trung bình thứ hạng 50/50 trên GPU

Đổi xác suất của từng mô hình thành vị trí trong danh sách đã sắp xếp. Nếu
nhiều giao dịch có cùng điểm, chúng nhận hạng trung bình của nhóm. Chốt trọng
số 50/50 trước khi xem nhãn và không huấn luyện mô hình tầng hai.

In [4]:
device = torch.device('cuda:0')

def fractional_ranks(values):
    ordered, positions = torch.sort(values, stable=True)
    _, counts = torch.unique_consecutive(ordered, return_counts=True)
    ends = counts.cumsum(dim=0).to(dtype=torch.float64)
    average = ends - (counts - 1).to(dtype=torch.float64) / 2
    ranks = torch.empty_like(values, dtype=torch.float64)
    ranks[positions] = torch.repeat_interleave(average, counts, output_size=len(values))
    return ranks / len(values)

example = torch.tensor([3.0, 1.0, 3.0, 2.0], device=device)
expected = torch.tensor([0.875, 0.25, 0.875, 0.5], device=device, dtype=torch.float64)
torch.testing.assert_close(fractional_ranks(example), expected)

started = perf_counter()
model_a = torch.as_tensor(lightgbm_oof, device=device)
model_b = torch.as_tensor(xgboost_oof, device=device)
combined = (fractional_ranks(model_a) + fractional_ranks(model_b)).mul_(0.5)
torch.cuda.synchronize(device)
rank_seconds = perf_counter() - started
oof = combined.cpu().numpy()
print(f'Thiết bị gộp: {torch.cuda.get_device_name(device)} | {rank_seconds:.3f} giây')

Thiết bị gộp: NVIDIA GeForce RTX 4060 Laptop GPU | 0.029 giây


## 5. Kết quả

Đo ROC AUC trên toàn bộ dự đoán giữ lại, xem từng fold và so trực tiếp với
hai mô hình gốc vừa tạo điểm. Báo riêng thời gian gộp và thời gian tạo OOF
của hai thành phần. Điểm gộp là thứ hạng chuẩn hóa để so sánh ROC AUC.

In [5]:
MODEL_SLUG = 'rank-blend'
device_name = f'{torch.cuda.get_device_name(0)} (CUDA rank averaging)'
for source, scores, info in [('lightgbm-te', lightgbm_oof, lightgbm_metrics),
                             ('xgboost-te', xgboost_oof, xgboost_metrics)]:
    measured = roc_auc_score(y, scores)
    if not np.isclose(measured, info['roc_auc'], rtol=0, atol=1e-12):
        raise ValueError(f'Điểm OOF không khớp báo cáo: {source}')
    print(f'{source}: ROC AUC {measured:.6f}')
fold_metrics = [
    {'fold': fold + 1, 'roc_auc': float(roc_auc_score(y[val], oof[val])),
     'validation_rows': len(val), 'device': device_name, 'fit_seconds': 0.0}
    for fold, (_, val) in enumerate(splits)
]
metrics = {
    'model': MODEL_SLUG, 'status': 'completed', 'smoke': False, 'models_use_llm': False,
    'rows': len(y), 'positives': int(y.sum()), 'train_sha256': DATA_HASH,
    'split_sha256': SPLIT_HASH, 'roc_auc': float(roc_auc_score(y, oof)),
    'average_precision': float(average_precision_score(y, oof)), 'fit_seconds': 0.0,
    'rank_seconds': rank_seconds, 'wall_seconds': rank_seconds,
    'reference_fit_seconds': sum(m['fit_seconds'] for m in (lightgbm_metrics, xgboost_metrics)),
    'reference_wall_seconds': sum(m['wall_seconds'] for m in (lightgbm_metrics, xgboost_metrics)),
    'reference_auc': {'lightgbm-te': lightgbm_metrics['roc_auc'],
                      'xgboost-te': xgboost_metrics['roc_auc']},
    'folds': fold_metrics,
}
output = ROUND2_DIR / 'results'
target = output / f'{MODEL_SLUG}.json'
temp = target.with_suffix('.tmp')
temp.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
temp.replace(target)
path = output / f'{MODEL_SLUG}-oof.npz'
stage = path.with_suffix('.tmp')
with stage.open('wb') as stream:
    np.savez(stream, ids=ids, fold=assignment, prediction=oof,
             train_sha256=DATA_HASH, split_sha256=SPLIT_HASH)
stage.replace(path)
for record in fold_metrics:
    print(f"Fold {record['fold']}: ROC AUC {record['roc_auc']:.6f}")
print(f"ROC AUC OOF: {metrics['roc_auc']:.6f}")
print(f"AP: {metrics['average_precision']:.6f} | Gộp hạng: {rank_seconds:.3f}s")

lightgbm-te: ROC AUC 0.948323
xgboost-te: ROC AUC 0.948311
Fold 1: ROC AUC 0.947197
Fold 2: ROC AUC 0.949820
Fold 3: ROC AUC 0.953542
Fold 4: ROC AUC 0.948956
Fold 5: ROC AUC 0.943025
ROC AUC OOF: 0.948474
AP: 0.518606 | Gộp hạng: 0.029s
